# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [15]:
%load_ext dotenv
%dotenv ../05_src/.secrets

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

## Imports 
<b><i>Separated imports for manageability.<i></b>

In [16]:
# Imports
import os, json
from openai import OpenAI
from pydantic import BaseModel
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

## Load API Key

In [17]:
# Load API Key

client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
api_key='any value',
default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})

## Load PDF dynamically 
<i>(pdf must exist in "documents/ folder"). This was done to test loading of other pdf files.</i>
<i>
- I used Chunking to handle loading .pdf documents to handle documents of any size as future documents may be larger than expected.
- I also chunked the PDF but concatenated the chunks back together for simplicity as this suits my use case.
- I did not use "overlap" in the chunking since I am just concatenating everything back together anyway.
</i>

In [18]:
# Load PDF

def load_pdf(fname):
    PDF_PATH = f"documents/{fname}"
    loader = PyPDFLoader(PDF_PATH)
    docs = loader.load()
    
    # Create text from docs
    text = "\n".join(p.page_content for p in docs)
    print(f"Loaded {len(docs)} page(s). Total characters: {len(text):,}")
    
    # Optional: chunk for robustness (but reassemble for single summary)
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=2000,
    )
    chunks = splitter.split_text(text)
    text = "\n".join(chunks)  # Reassemble with clear boundaries
    print(f"Split into {len(chunks)} chunk(s)")
    
    return text


document_text = load_pdf("managing_oneself.pdf")

Loaded 13 page(s). Total characters: 51,451
Split into 29 chunk(s)


## Call OpenAPI Chat
<b><i>I decided to use OpenAPI Chat because Part 2 of the assignment is to build a chatbot. This ensures code cosistency.</i></b>

<i>This code sends an article to the model, asks for a structured summary and metadata, and prints the result as JSON.

This is how the code works:

- Defines two Pydantic models:

   - ArticleSummary with fields: Author, Title, Relevance, Summary, Tone.
   
   - ArticleSummaryTokensUsed that extends ArticleSummary and adds InputTokens and OutputTokens (optional ints).

   This was done to keep the API contract separate from internal tracking needs.

- I used the chat completions API:

   - Structured output: —no string parsing needed.
   - Handles Multi-role messaging: developer + user roles let us separate instructions from content.
   - GPT-4o-mini is cheap for summarization tasks.
   - Built-in token tracking: response.usage gives us prompt/completion tokens without extra work.

- Constructs a dictionary with the extracted fields plus token usage and prints it as pretty JSON.
   - Clarity: We know exactly what json.dumps() is outputting without inspecting the model.
</i> 


In [19]:
class ArticleSummary(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: str

class ArticleSummaryTokensUsed(ArticleSummary):
    InputTokens: int | None = None
    OutputTokens: int | None = None

DEVELOPER_PROMPT = """
You are an expert academic summarization assistant.

1. Author: extract the author.
2. Title: extract the title.
3. Relevance: write one paragraph on why this article matters for an AI professional's
   professional development.
4. Summary: write a concise summary, no longer than 1000 tokens, written in a Formal
   Academic Writing tone.
5. Tone: set to "Formal Academic Writing".
"""

USER_PROMPT = f"""Article content:

{document_text}
"""

response = client.chat.completions.parse(
    model="gpt-4o-mini",
    messages=[
        {"role": "developer", "content": DEVELOPER_PROMPT},
        {"role": "user", "content": USER_PROMPT},
    ],
    response_format=ArticleSummary,
)

parsed = response.choices[0].message.parsed
parsed_output = ArticleSummaryTokensUsed(
    **parsed.model_dump(),
    InputTokens=response.usage.prompt_tokens,
    OutputTokens=response.usage.completion_tokens,
)


formatted = {
"Author": parsed_output.Author,
"Title": parsed_output.Title,
"Relevance": parsed_output.Relevance,
"Summary": parsed_output.Summary,
"Tone": parsed_output.Tone,
"InputTokens": parsed_output.InputTokens,
"OutputTokens": parsed_output.OutputTokens,
}
print(json.dumps(formatted, ensure_ascii=False, indent=2))

{
  "Author": "Peter F. Drucker",
  "Title": "Managing Oneself",
  "Relevance": "In today's rapidly evolving professional landscape, particularly within the realm of artificial intelligence, the concepts presented in Drucker's 'Managing Oneself' are essential for AI professionals looking to navigate their careers effectively. As knowledge workers, AI specialists must take accountability for their career trajectories, embodying the role of their own chief executive officers. This article highlights the importance of self-awareness, especially regarding one's strengths, learning methods, work preferences, and ethical values, which are crucial traits for cultivating innovation and maintaining effectiveness in a competitive field dominated by constant technological advancements. Embracing these principles not only fosters personal growth but also enhances one's contribution to organizational success in AI and beyond.",
  "Summary": "In 'Managing Oneself,' Peter F. Drucker articulates the c

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
